# Hugging Face Fundamentals — Lesson 2: Hugging Face Pipelines

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `02_HuggingFace_Pipeline.py` (same content, runnable without Jupyter).

**Task ID:** HF-002  |  **Folder:** `02_HuggingFace_Pipeline`


## Why pipelines?

Using a model normally means five steps: load the model, load the tokenizer, convert the text, run the network, convert the numbers back. The **pipeline** squeezes all of that into **one line**:

> `pipeline(task, model)` → a callable object that predicts for you.

## The hidden recipe

Every pipeline runs the same recipe in the background:

1. **tokenize** — turn text into number ids,
2. **forward** — run the neural network,
3. **post-process** — turn raw scores into labels / text / spans.

You control the dials (`device`, batch, `top_k`...), the pipeline does the rest.

**Step 1 — a sentiment pipeline** (first call downloads the model once, then it is cached on disk):


In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=-1)
print(sentiment("I love this product, it works perfectly!"))


`device=-1` means *run on CPU* (safe default). Remove it or set 0 on a machine with a GPU.

**Step 2 — batch:** hand over a *list* of texts in one call:


In [ ]:
texts = [
    "I love this product!",
    "The delivery took two weeks and the box was damaged.",
    "It is fine, nothing special.",
]
for text, r in zip(texts, sentiment(texts)):
    print(f"{r['label']:<9} ({r['score']:.2f})  {text!r}")


## The task zoo

Same one-liner, different tasks. Each loads its own small model:


In [ ]:
classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)
print(classifier("The acting was brilliant but the ending felt rushed.", top_k=2))


In [ ]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1,
)
result = zero_shot(
    "The central bank raised interest rates again this morning.",
    candidate_labels=["finance", "sports", "health"],
)
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<10} {score:.3f}")


In [ ]:
from transformers import GenerationConfig

generator = pipeline("text-generation", model="distilgpt2", device=-1)
config = GenerationConfig(max_new_tokens=25, do_sample=True, temperature=0.9, top_k=50)
print(generator("Once upon a time", generation_config=config)[0]["generated_text"])


## Try it yourself

1. Classify 5 of your own sentences with the sentiment pipeline.
2. Zero-shot: classify a news headline with labels `politics, sports, weather`.
3. Change `--model` to a different id on the Hub — the pipeline API does not change.

## Common pitfalls

- **Wrong model for the task** — a *causal* model (GPT-2) cannot do classification. Match task and model family.
- **"CUDA out of memory"** — set `device=-1` or use a smaller model.
- **Download takes a while the first time** — normal; afterwards it is cached.

## Summary

- `pipeline(task, model)` = tokenizer + model + post-processing in one object.
- Batches: pass a list; results come back in order.
- Tasks and model families must match.

**Next lesson:** HF-003 — Tokenizer.  |  Extra reading: `../resources/reference_links.md`
